In [17]:
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd

In [6]:
PROJ_DIR = Path.cwd().parent.parent
DATA_DIR = PROJ_DIR / 'data'
fp = DATA_DIR / 'calhabmap.csv'
assert fp.exists()

### ETL

* Load
* Quick look assess cleanup requirements
* Note columns of interest

In [25]:
df_raw = pd.read_csv(fp)

In [26]:
df_raw.head().T

,0,1,2,3,4
FID,calhabmap.fid--342fab54_19a6c67ae25_16b9,calhabmap.fid--342fab54_19a6c67ae25_16ba,calhabmap.fid--342fab54_19a6c67ae25_16bb,calhabmap.fid--342fab54_19a6c67ae25_16bc,calhabmap.fid--342fab54_19a6c67ae25_16bd
latitude_degrees_north,35.17,34.008,40.7103,34.008,35.17
longitude_degrees_east,-120.741,-118.499,-124.2366,-118.499,-120.741
depth_m,0.5,NaN,NaN,NaN,0.5
sampleid,CP180709,403,SBF_002,165,CP110927
location_code,CPP,SMP,HSB,SMP,CPP
time_utc,2018-07-09T22:00:00,2016-05-02T18:00:00,2020-01-27T18:30:00,2011-09-04T19:00:00,2011-09-27T22:30:00
temp_degree_c,16.9,15.8,NaN,19.5,17.0
air_temp_degree_c,NaN,NaN,NaN,NaN,NaN
salinity,NaN,NaN,NaN,NaN,NaN


In [36]:
CORE_RENAME = {
    "sampleid": "sample_id",
    "location_code": "site_code",
    "location_name": "site_name",
    "latitude_degrees_north": "lat",
    "longitude_degrees_east": "lon",
}

SCIENCE_COLS = [
    # ancillary
    "temp_degree_c", "chl1_mg_m3", "chl2_mg_m3", "avg_chloro_mg_m3",
    
    "volume_settled_for_counting_ml",
    # totals
    "total_phytoplankton_cells_l",
]





In [47]:
    # --- Standardize core columns
    df = df_raw.rename(columns=CORE_RENAME).copy()

In [48]:
df["time_utc"] = pd.to_datetime(df["time_utc"], errors="coerce", utc=True)


# Apply cutoff
cutoff = pd.Timestamp("2024-03-06", tz="UTC")
df = df[df["time_utc"] >= cutoff]

print("Rows after cutoff:", len(df))
print("Date range:", df["time_utc"].min(), "→", df["time_utc"].max())


Rows after cutoff: 99
Date range: 2024-08-14 14:31:00+00:00 → 2025-11-05 15:35:00+00:00


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99 entries, 1156 to 5715
Data columns (total 43 columns):
 #   Column                                        Non-Null Count  Dtype              
---  ------                                        --------------  -----              
 0   FID                                           99 non-null     object             
 1   lat                                           99 non-null     float64            
 2   lon                                           99 non-null     float64            
 3   depth_m                                       39 non-null     float64            
 4   sample_id                                     99 non-null     object             
 5   site_code                                     99 non-null     object             
 6   time_utc                                      99 non-null     datetime64[ns, UTC]
 7   temp_degree_c                                 94 non-null     float64            
 8   air_temp_degree_c     

In [44]:
def find_cellcount_columns(columns: Iterable[str]) -> list[str]:
    return [c for c in columns if isinstance(c, str) and c.endswith(CELLCOUNT_SUFFIX)]

KEEP_COLS = SCIENCE_COLS + [v for v in CORE_RENAME.values()] + 

In [45]:
df = df[KEEP_COLS].copy()

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99 entries, 1156 to 5715
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   temp_degree_c                   94 non-null     float64
 1   chl1_mg_m3                      45 non-null     float64
 2   chl2_mg_m3                      45 non-null     float64
 3   avg_chloro_mg_m3                45 non-null     float64
 4   volume_settled_for_counting_ml  33 non-null     float64
 5   total_phytoplankton_cells_l     65 non-null     float64
 6   sample_id                       99 non-null     object 
 7   site_code                       99 non-null     object 
 8   site_name                       99 non-null     object 
 9   lat                             99 non-null     float64
 10  lon                             99 non-null     float64
 11  total_phytoplankton_cells_l     65 non-null     float64
dtypes: float64(9), object(3)
memory usage: